# 任务2.3 对粮食产量进行统计分析：NumPy 2.0兼容与细节补充

编制日期：2026-10-12，适用第04周“NumPy数组计算基础·第二部分”。教师确认2.2已讲完；2.3本次准备，不记为已授课。

- **来源**：知识准备/项目2/code/任务2.3 对粮食产量进行统计分析.ipynb。原件43个代码单元逐一对应，`原代码N`和单元元数据可追溯；原件只读、未运行。
- **目标**：掌握数组保存读取、排序索引、去重重复和统计，能说明axis、修改原数组及统计口径。
- **先修**：2.2数组运算、广播，Python索引切片、函数调用；补充`with`、`Path`时先解释后使用。
- **输入**：教材内嵌数学例子与新模拟小数据，无外部CSV，无真实粮食产量。本任务属于知识准备，不冒充综合业务案例。
- **依赖/运行**：现有Python 3、NumPy；从本Notebook所在的任务2.3子目录启动Jupyter，选课程内核，重启后从上至下运行。不要在原教材code目录运行。
- **输出**：只在当前任务2.3的`outputs/`写5个演示文件；重跑会覆盖同名教学输出。无网络、数据库或环境修改。
- **兼容审查**：原43个单元没有使用NumPy 2.0已移除或正式弃用的API。`seed/randint`属于仍支持的旧接口，新代码采用Generator。其余修订属于精度、资源管理或讲解质量问题。
- **验证边界**：按NumPy 2.0官方文档审查；本机实际运行版本以同目录验证记录为准，不能等同于已在2.0.0或Windows机房运行。

建议基础课堂沿“问题导入10%→教师演示25%→跟练35%→独立预测20%→解释结果10%”组织；按剩余课时比例安排，不预设每学时分钟数。基础必做读写、排序、axis、均值和极值；多键排序、稳定去重、缺失值、累计积溢出为拓展。

原编号代码均为教师演示；标为“补充”的单元为额外知识点。课后任选两个预测问题提交结果和解释，评价结果正确、口径清楚、能说明是否修改原数据。

依据：[NumPy 2.0迁移指南](https://numpy.org/doc/2.0/numpy_2_0_migration_guide.html)。

## 1. 数组文件的保存与读取（原代码1—9，基础）

业务导入：有小数的产量保存后，重新读入还能保持一致吗？`.npy`保存一个数组及其shape/dtype，`.npz`保存多个数组；文本便于人阅读，但精度、分隔符、编码和表头要明确。

补足语法：`Path.cwd()`返回内核当前工作目录；`/`拼接路径，`mkdir(..., exist_ok=True)`创建输出目录。请从任务2.3目录启动，下面检查Notebook文件名以防误写到别处。读写都在新备课副本，不接触原教材tmp。

In [1]:
from pathlib import Path
import numpy as np

NOTEBOOK_NAME = '任务2.3 对粮食产量进行统计分析_新版NumPy.ipynb'
TASK_DIR = Path.cwd()
if not (TASK_DIR / NOTEBOOK_NAME).is_file():
    raise RuntimeError('请从任务2.3子目录启动Jupyter，重启内核后运行。')
OUTPUT_DIR = TASK_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('NumPy版本：', np.__version__)
print('输出目录：', OUTPUT_DIR)

NumPy版本： 2.5.2
输出目录： /private/var/folders/qj/yrdc2hrx3mg1qw023rfvp3600000gn/T/week04-task23-xfmez4hr/work/outputs


原代码1：`.npy`与输出路径。API仍支持，`../tmp`改为当前任务的outputs。

In [2]:
# 原代码1
import numpy as np  # 导入NumPy库
arr = np.arange(100).reshape(10, 10)
np.save(OUTPUT_DIR / 'save_arr.npy', arr, allow_pickle=False)
print('保存的数组为：\n', arr)

保存的数组为：
 [[ 0  1  2  3  4  5  6  7  8  9]
 [10 11 12 13 14 15 16 17 18 19]
 [20 21 22 23 24 25 26 27 28 29]
 [30 31 32 33 34 35 36 37 38 39]
 [40 41 42 43 44 45 46 47 48 49]
 [50 51 52 53 54 55 56 57 58 59]
 [60 61 62 63 64 65 66 67 68 69]
 [70 71 72 73 74 75 76 77 78 79]
 [80 81 82 83 84 85 86 87 88 89]
 [90 91 92 93 94 95 96 97 98 99]]


原代码2—3：`.npz`多个数组。`arange`小数步长仍支持，但点数可能受浮点误差影响；需要固定点数时用linspace。

In [3]:
# 原代码2
arr1 = np.array([[1, 2, 3], [4, 5, 6]])
# 指定10个等距点，包含0.9；替换小数步长arange，不依赖步长累积。
arr2 = np.linspace(0.0, 0.9, num=10)
# 用明确键名；位置参数的arr_0/arr_1仍受支持。
np.savez(OUTPUT_DIR / 'savez_arr.npz', arr1=arr1, arr2=arr2)
print('保存的数组arr1为：\n', arr1)

保存的数组arr1为：
 [[1 2 3]
 [4 5 6]]


In [4]:
# 原代码3
print('保存的数组arr2为：', arr2)

保存的数组arr2为： [0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9]


原代码4：读取单数组。本例只有数值，不需要pickle。

In [5]:
# 原代码4
loaded_data = np.load(OUTPUT_DIR / 'save_arr.npy', allow_pickle=False)
print('读取的数组为：\n', loaded_data)

读取的数组为：
 [[ 0  1  2  3  4  5  6  7  8  9]
 [10 11 12 13 14 15 16 17 18 19]
 [20 21 22 23 24 25 26 27 28 29]
 [30 31 32 33 34 35 36 37 38 39]
 [40 41 42 43 44 45 46 47 48 49]
 [50 51 52 53 54 55 56 57 58 59]
 [60 61 62 63 64 65 66 67 68 69]
 [70 71 72 73 74 75 76 77 78 79]
 [80 81 82 83 84 85 86 87 88 89]
 [90 91 92 93 94 95 96 97 98 99]]


原代码5—6：读取多数组。`np.load(.npz)`返回NpzFile，使用with及时关闭；`with ... as archive`把对象绑定给变量。

In [6]:
# 原代码5
# with代码块结束时关闭NpzFile；取出的数值数组仍可在后续单元使用。
with np.load(OUTPUT_DIR / 'savez_arr.npz', allow_pickle=False) as archive:
    print('归档中的键名：', archive.files)
    loaded_arr1 = archive['arr1']
    loaded_arr2 = archive['arr2']
print('读取的数组arr1为：\n', loaded_arr1)

归档中的键名： ['arr1', 'arr2']
读取的数组arr1为：
 [[1 2 3]
 [4 5 6]]


In [7]:
# 原代码6
print('读取的数组arr2为：', loaded_arr2)

读取的数组arr2为： [0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9]


### 补充：保存格式对比（基础）

二进制通常保留dtype/shape；文本读入默认float。npz位置参数arr_0/arr_1仍支持，命名键只是更清晰。这里比较刚刚写读的数据。

In [8]:
print('npy类型与形状：', loaded_data.dtype, loaded_data.shape)
print('npz数值一致：', np.array_equal(loaded_arr1, arr1), np.array_equal(loaded_arr2, arr2))

npy类型与形状： int64 (10, 10)
npz数值一致： True True


原代码7—9：文本读写。`reshape(4, -1)`自动推算6列；0.5可被二进制精确表示，与0.1不同。`%d`不会自动保留小数。

In [9]:
# 原代码7
arr = np.arange(0, 12, 0.5).reshape(4, -1)
print('创建的数组为：\n', arr)

创建的数组为：
 [[ 0.   0.5  1.   1.5  2.   2.5]
 [ 3.   3.5  4.   4.5  5.   5.5]
 [ 6.   6.5  7.   7.5  8.   8.5]
 [ 9.   9.5 10.  10.5 11.  11.5]]


In [10]:
# 原代码8
# 原fmt='%d'仍可运行，但会截去0.5等小数；本例以%.1f保留原数据。
np.savetxt(OUTPUT_DIR / 'arr.txt', arr, fmt='%.1f', delimiter=',', encoding='utf-8')
loaded_data = np.loadtxt(OUTPUT_DIR / 'arr.txt', delimiter=',', encoding='utf-8')
print('读取的数组为：\n', loaded_data)
print('读写数值一致：', np.array_equal(loaded_data, arr))

读取的数组为：
 [[ 0.   0.5  1.   1.5  2.   2.5]
 [ 3.   3.5  4.   4.5  5.   5.5]
 [ 6.   6.5  7.   7.5  8.   8.5]
 [ 9.   9.5 10.  10.5 11.  11.5]]
读写数值一致： True


In [11]:
# 原代码9
loaded_data = np.genfromtxt(OUTPUT_DIR / 'arr.txt', delimiter=',', encoding='utf-8')
print('读取的数组为：\n', loaded_data)

读取的数组为：
 [[ 0.   0.5  1.   1.5  2.   2.5]
 [ 3.   3.5  4.   4.5  5.   5.5]
 [ 6.   6.5  7.   7.5  8.   8.5]
 [ 9.   9.5 10.  10.5 11.  11.5]]


### 补充：看见%d丢失小数（基础）

原写法不是NumPy 2.0移除；是格式与数据不匹配。先预测0.5和1.5写入后是什么，再对照。%.1f适合本例，不保证任意float都无损；一般float64往返可用%.18e或保存npy。

In [12]:
np.savetxt(OUTPUT_DIR / 'arr_整数格式对照.txt', arr, fmt='%d', delimiter=',', encoding='utf-8')
lost = np.loadtxt(OUTPUT_DIR / 'arr_整数格式对照.txt', delimiter=',', encoding='utf-8')
print('原第一行：', arr[0])
print('%d读取的第一行：', lost[0])
print('发生小数损失：', not np.array_equal(lost, arr))

原第一行： [0.  0.5 1.  1.5 2.  2.5]
%d读取的第一行： [0. 0. 1. 1. 2. 2.]
发生小数损失： True


### 补充：表头、空值与读取选择（拓展）

新模拟年度数据，单位万吨。loadtxt适合规则纯数值文本；genfromtxt可识别空字段，默认浮点空值为nan。表头需skip_header=1；空值留待业务规则，不自动填0。文件编码utf-8-sig兼容UTF-8 BOM。

In [13]:
text_path = OUTPUT_DIR / '模拟产量_含缺失.csv'
text_path.write_text('年份,模拟产量(万吨)\n2021,10.5\n2022,\n2023,12.0\n', encoding='utf-8-sig')
missing_table = np.genfromtxt(text_path, delimiter=',', skip_header=1, encoding='utf-8-sig', ndmin=2)
print('保留缺失值：\n', missing_table)
print('缺失个数：', np.isnan(missing_table[:, 1]).sum())

保留缺失值：
 [[2021.    10.5]
 [2022.     nan]
 [2023.    12. ]]
缺失个数： 1


## 2. 排序与间接排序（原代码10—17，基础）

原代码10：旧随机接口仍支持，新代码采用`default_rng(42).integers(...)`。相同种子在新旧算法下不保证得到同一序列；不拿课本旧输出逐项比对。重新创建rng会重新开始；连续调用同一rng会推进状态。

In [14]:
# 原代码10
# seed/randint仍支持；Generator对象不重置其他单元的全局随机状态。
rng = np.random.default_rng(42)
arr = rng.integers(1, 10, size=10)  # [1, 10)，不包含10
print('创建的数组为：', arr)

创建的数组为： [1 7 6 4 4 8 1 7 2 1]


原代码11：`arr.sort()`原地修改并返回None；不要写`arr = arr.sort()`。

In [15]:
# 原代码11
arr.sort()  # 直接排序
print('排序后数组为：', arr)

排序后数组为： [1 1 1 2 4 4 6 7 7 8]


### 补充：原地排序与新数组（基础）

预测x排序后是否变，result是什么；np.sort返回排序副本。不能单独排序产量列后继续配原年份，必须用统一索引。

In [16]:
x = np.array([3, 1, 2])
sorted_x = np.sort(x)
print('np.sort后原x：', x, '新数组：', sorted_x)
result = x.sort()
print('x.sort后原x：', x, '返回值：', result)

np.sort后原x： [3 1 2] 新数组： [1 2 3]
x.sort后原x： [1 2 3] 返回值： None


原代码12—14：二维排序。`axis=1`在每行内排序，`axis=0`在每列内排序；教材先行后列是连续修改同一个数组。

In [17]:
# 原代码12
rng = np.random.default_rng(42)
arr = rng.integers(1, 10, size=(3, 3))
print('创建的数组为：\n', arr)
original_2d = arr.copy()  # 给独立比较两种axis保留一份副本

创建的数组为：
 [[1 7 6]
 [4 4 8]
 [1 7 2]]


In [18]:
# 原代码13
arr.sort(axis=1)  # 每一行内部排序，列之间移动数值；原数组被修改
print('每行排序后：\n', arr)

每行排序后：
 [[1 6 7]
 [4 4 8]
 [1 2 7]]


In [19]:
# 原代码14
# 对原代码13已经排过行的结果继续按列排序；不是回到原始随机数组。
arr.sort(axis=0)
print('再对每列排序后：\n', arr)

再对每列排序后：
 [[1 2 7]
 [1 4 7]
 [4 6 8]]


### 补充：分别从原数组比较axis（基础）

教材连续两次sort受前一步影响，下面每次从同一原始数组取副本；这也避免把值排序误认为业务行排序。

In [20]:
print('原始二维数组：\n', original_2d)
print('独立每行排序：\n', np.sort(original_2d, axis=1))
print('独立每列排序：\n', np.sort(original_2d, axis=0))

原始二维数组：
 [[1 7 6]
 [4 4 8]
 [1 7 2]]
独立每行排序：
 [[1 6 7]
 [4 4 8]
 [1 2 7]]
独立每列排序：
 [[1 4 2]
 [1 7 6]
 [4 7 8]]


原代码15—16：argsort返回索引，不是排序后的值；用`arr[索引]`取值。

In [21]:
# 原代码15
arr = np.array([2, 3, 6, 8, 0, 7])
print('创建的数组为：', arr)

创建的数组为： [2 3 6 8 0 7]


In [22]:
# 原代码16
print('排序后索引数组为：', arr.argsort())  # 返回值为排序后元素的索引数组

排序后索引数组为： [4 0 1 2 5 3]


### 补充：排序索引同步业务列（基础）

下面是新模拟数据，单位万吨。索引[1,2,0]让年份与产量一起移动；增长按时间升序计算，以前一年为基期。

In [23]:
years = np.array([2023, 2021, 2022])
yields = np.array([12.0, 10.0, 11.0])
order = np.argsort(years)
years_sorted = years[order]
yields_sorted = yields[order]
print('排序索引：', order)
print('年与模拟产量：', list(zip(years_sorted, yields_sorted)))
print('后两年同比百分数：', np.diff(yields_sorted) / yields_sorted[:-1] * 100)

排序索引： [1 2 0]
年与模拟产量： [(np.int64(2021), np.float64(10.0)), (np.int64(2022), np.float64(11.0)), (np.int64(2023), np.float64(12.0))]
后两年同比百分数： [10.          9.09090909]


### 补充：同值排序保持先后（拓展）

kind="stable"在2.0可用，要求同值记录保留原先后。稳定降序不能简单把整个升序索引[::-1]，那会把同值组内部也翻转。下例小浮点数用负值排序得到稳定降序；不能随意推广到无符号整数。

In [24]:
scores = np.array([2.0, 1.0, 2.0, 1.0])
labels = np.array(['甲', '乙', '丙', '丁'])
asc = np.argsort(scores, kind='stable')
desc = np.argsort(-scores, kind='stable')
print('稳定升序：', labels[asc])
print('稳定降序：', labels[desc])

稳定升序： ['乙' '丁' '甲' '丙']
稳定降序： ['甲' '丙' '乙' '丁']


原代码17：lexsort多键排序。本例主键c没有重复，无法展示次键的作用；后面的补充用同值主键说明。

In [25]:
# 原代码17
a = np.array([3, 2, 6, 4, 5])
b = np.array([50, 30, 40, 20, 10])
c = np.array([400, 300, 600, 100, 200])
# keys是一个元组；lexsort还支持axis参数，不能说“只接收一个参数”。
# 最后一个键c是主键，同值时先比较b，再比较a；本例c无重复。
d = np.lexsort((a, b, c))
print('排序后数组为：\n', list(zip(a[d], b[d], c[d])))

排序后数组为：
 [(np.int64(4), np.int64(20), np.int64(100)), (np.int64(5), np.int64(10), np.int64(200)), (np.int64(2), np.int64(30), np.int64(300)), (np.int64(3), np.int64(50), np.int64(400)), (np.int64(6), np.int64(40), np.int64(600))]


### 补充：让lexsort的次键真正起作用（拓展）

主键地区编号相同时比较年份；lexsort((年份,地区))最后一个键优先。原注释“只接收一个参数”已改正，签名为lexsort(keys, axis=-1)。

In [26]:
region = np.array([2, 1, 1, 2])
year = np.array([2022, 2023, 2021, 2021])
lex_order = np.lexsort((year, region))
print('先地区、再年份：', list(zip(region[lex_order], year[lex_order])))

先地区、再年份： [(np.int64(1), np.int64(2021)), (np.int64(1), np.int64(2023)), (np.int64(2), np.int64(2021)), (np.int64(2), np.int64(2022))]


## 3. 去重与重复（原代码18—27，基础）

原代码18—20：`np.unique`默认返回排序后的唯一值；中文按字符编码比较，不是拼音顺序，也不是首次出现顺序。

In [27]:
# 原代码18
names = np.array(['小明', '小黄', '小花', '小明', '小花', '小兰', '小白'])
print('创建的数组为：', names)

创建的数组为： ['小明' '小黄' '小花' '小明' '小花' '小兰' '小白']


In [28]:
# 原代码19
print('去重后的数组为：', np.unique(names))

去重后的数组为： ['小兰' '小明' '小白' '小花' '小黄']


In [29]:
# 原代码20
# 对本例一维中文字符串，值与顺序相同；不是通用的NumPy去重替代。
# sorted(set(...))返回列表，np.unique返回ndarray。
print('去重后的数组为：', sorted(set(names)))

去重后的数组为： [np.str_('小兰'), np.str_('小明'), np.str_('小白'), np.str_('小花'), np.str_('小黄')]


原代码21—22：数值去重。不等于发现并处理业务重复记录，业务中还需明确重复判定键。

In [30]:
# 原代码21
# 创建数值型数组
ints = np.array([1, 2, 3, 4, 4, 5, 6, 6, 7, 8, 8, 9, 10]) 
print('创建的数组为：', ints)

创建的数组为： [ 1  2  3  4  4  5  6  6  7  8  8  9 10]


In [31]:
# 原代码22
print('去重后的数组为：', np.unique(ints))

去重后的数组为： [ 1  2  3  4  5  6  7  8  9 10]


### 补充：频数与首次出现顺序（拓展）

return_counts给频数；return_index给每个唯一值首次出现位置。把这些位置排序，才恢复首次出现顺序。sorted(set(...))仅对本例一维同类可哈希值有效；多维、nan、输出类型等行为不同。

In [32]:
values, first, counts = np.unique(names, return_index=True, return_counts=True)
print('按默认顺序的名字与频数：', list(zip(values, counts)))
print('按首次出现顺序：', names[np.sort(first)])

按默认顺序的名字与频数： [(np.str_('小兰'), np.int64(1)), (np.str_('小明'), np.int64(2)), (np.str_('小白'), np.int64(1)), (np.str_('小花'), np.int64(2)), (np.str_('小黄'), np.int64(1))]
按首次出现顺序： ['小明' '小黄' '小花' '小兰' '小白']


原代码23—24：`tile`重复整个模式；`repeat`重复每个元素或指定轴的切片。

In [33]:
# 原代码23
arr = np.arange(5)
print('创建的数组为：', arr)

创建的数组为： [0 1 2 3 4]


In [34]:
# 原代码24
print('重复后数组为：', np.tile(arr, 3))  # 对数组进行重复

重复后数组为： [0 1 2 3 4 0 1 2 3 4 0 1 2 3 4]


原代码25—27：二维repeat。axis=0重复行，axis=1重复列；不要只靠“横轴/纵轴”记忆。

In [35]:
# 原代码25
rng = np.random.default_rng(42)
arr = rng.integers(0, 10, size=(3, 3))
print('创建的数组为：\n', arr)

创建的数组为：
 [[0 7 6]
 [4 4 8]
 [0 6 2]]


In [36]:
# 原代码26
print('重复整行后的数组为：\n', arr.repeat(2, axis=0))  # 每行连续重复2次

重复整行后的数组为：
 [[0 7 6]
 [0 7 6]
 [4 4 8]
 [4 4 8]
 [0 6 2]
 [0 6 2]]


In [37]:
# 原代码27
print('重复整列后的数组为：\n', arr.repeat(2, axis=1))  # 每列连续重复2次

重复整列后的数组为：
 [[0 0 7 7 6 6]
 [4 4 4 4 8 8]
 [0 0 6 6 2 2]]


### 补充：tile与repeat的差异（基础）

先预测[1,2]各重复两次的顺序。二维tuple指定各轴重复倍数；重复数据不会增加真实观测数，不用来“扩充样本”。

In [38]:
pattern = np.array([1, 2])
print('tile整个模式：', np.tile(pattern, 2))
print('repeat每个元素：', np.repeat(pattern, 2))
print('二维tile形状：', np.tile(np.array([[1, 2], [3, 4]]), (2, 1)).shape)

tile整个模式： [1 2 1 2]
repeat每个元素： [1 1 2 2]
二维tile形状： (4, 2)


## 4. 统计与归约（原代码28—40，基础）

原代码28—31：axis表示被归约的维度。形状(4,5)去掉axis=0后剩(5,)，去掉axis=1后剩(4,)。

In [39]:
# 原代码28
arr = np.arange(20).reshape(4, 5)
print('创建的数组为：\n', arr)

创建的数组为：
 [[ 0  1  2  3  4]
 [ 5  6  7  8  9]
 [10 11 12 13 14]
 [15 16 17 18 19]]


In [40]:
# 原代码29
print('数组的和为：', np.sum(arr))  # 计算数组的和

数组的和为： 190


In [41]:
# 原代码30
print('沿axis=0归约，得到每列的和：', arr.sum(axis=0))

沿axis=0归约，得到每列的和： [30 34 38 42 46]


In [42]:
# 原代码31
print('沿axis=1归约，得到每行的和：', arr.sum(axis=1))

沿axis=1归约，得到每行的和： [10 35 60 85]


原代码32—34：均值。未写axis时统计全部元素；同一张业务表各列应有可比单位，否则不能对全表算一个总均值。

In [43]:
# 原代码32
print('数组的均值为：', np.mean(arr))  # 计算数组均值

数组的均值为： 9.5


In [44]:
# 原代码33
print('每列的均值：', arr.mean(axis=0))

每列的均值： [ 7.5  8.5  9.5 10.5 11.5]


In [45]:
# 原代码34
print('每行的均值：', arr.mean(axis=1))

每行的均值： [ 2.  7. 12. 17.]


### 补充：axis与keepdims连接广播（基础）

新模拟数组行是年份、列是品种，单位万吨。axis=0统计各品种跨年均值；axis=1统计每年这两个品种的合计（仅示范，两者合计不是全国全部粮食）。keepdims=True保留长度为1的轴，便于2.2广播。

In [46]:
production = np.array([[10.0, 20.0], [12.0, 22.0], [14.0, 24.0]])
column_mean = production.mean(axis=0, keepdims=True)
print('各品种跨年均值及形状：', column_mean, column_mean.shape)
print('每年两个品种合计：', production.sum(axis=1))
print('各年产量减各品种均值：\n', production - column_mean)

各品种跨年均值及形状： [[12. 22.]] (1, 2)
每年两个品种合计： [30. 34. 38.]
各年产量减各品种均值：
 [[-2. -2.]
 [ 0.  0.]
 [ 2.  2.]]


原代码35—36：方差和标准差。NumPy默认ddof=0，即除以N；ddof=1除以N-1，常用于样本方差估计，需要N>1。

In [47]:
# 原代码35
print('全部20个数的总体标准差：', np.std(arr, ddof=0))

全部20个数的总体标准差： 5.766281297335398


In [48]:
# 原代码36
print('全部20个数的总体方差：', np.var(arr, ddof=0))

全部20个数的总体方差： 33.25


### 补充：ddof、单位与nan（拓展）

标准差与数据同单位（万吨），方差单位为万吨的平方。np.mean含nan会返回nan；nanmean只对非缺失观测求均值，要同时报告有效样本数，不能当成缺失已被修复。

In [49]:
sample = np.array([10.0, 12.0, 14.0])
print('总体方差、标准差：', np.var(sample, ddof=0), np.std(sample, ddof=0))
print('样本方差、标准差：', np.var(sample, ddof=1), np.std(sample, ddof=1))
valid = missing_table[:, 1]
print('直接均值：', np.mean(valid))
print('忽略缺失的均值及有效数：', np.nanmean(valid), np.count_nonzero(~np.isnan(valid)))

总体方差、标准差： 2.6666666666666665 1.632993161855452
样本方差、标准差： 4.0 2.0
直接均值： nan
忽略缺失的均值及有效数： 11.25 2


原代码37—40：极值与索引。argmin/argmax未写axis时返回展平位置；相同极值返回首次出现的索引。

In [50]:
# 原代码37
print('数组的最小值为：', np.min(arr))  # 计算数组最小值

数组的最小值为： 0


In [51]:
# 原代码38
print('数组的最大值为：', np.max(arr))  # 计算数组最大值

数组的最大值为： 19


In [52]:
# 原代码39
print('数组最小元素的展平索引：', np.argmin(arr))  # 未指定axis时按C顺序展平

数组最小元素的展平索引： 0


In [53]:
# 原代码40
print('数组最大元素的展平索引：', np.argmax(arr))

数组最大元素的展平索引： 19


### 补充：展平索引、二维坐标及并列极值（基础）

展平索引19对于(4,5)是(3,4)，从0开始。unravel_index还原坐标；argmax只能给首次最大值，argwhere可列出全部并列坐标。

In [54]:
flat_max = np.argmax(arr)
max_row, max_col = np.unravel_index(flat_max, arr.shape)
print('展平索引、坐标与值：', flat_max, (max_row, max_col), arr[max_row, max_col])
print('每列最大值的行索引：', np.argmax(arr, axis=0))
ties = np.array([[5, 1], [2, 5]])
print('首次最大值展平索引：', np.argmax(ties))
print('所有最大值坐标：\n', np.argwhere(ties == np.max(ties)))

展平索引、坐标与值： 19 (np.int64(3), np.int64(4)) 19
每列最大值的行索引： [3 3 3 3 3]
首次最大值展平索引： 0
所有最大值坐标：
 [[0 0]
 [1 1]]


## 5. 累计计算（原代码41—43，基础与拓展）

原代码41—43：cumsum不是总和，cumprod不是普通乘积；每个位置包含从开始到该位置的所有值。本例[2,...,9]不溢出，但大整数连乘可能溢出且不报错。

In [55]:
# 原代码41
arr = np.arange(2, 10)
print('创建的数组为：', arr)

创建的数组为： [2 3 4 5 6 7 8 9]


In [56]:
# 原代码42
print('数组元素的累计和为：', np.cumsum(arr))  # 计算所有元素的累计和

数组元素的累计和为： [ 2  5  9 14 20 27 35 44]


In [57]:
# 原代码43
print('数组元素的累计积为：\n', np.cumprod(arr, dtype=np.int64))  # 显式指定整数精度

数组元素的累计积为：
 [     2      6     24    120    720   5040  40320 362880]


### 补充：累计计算的axis、时间方向与增长因子（基础）

不写axis默认先展平；二维表须指定axis。下面是新模拟年度产量，单位万吨；累计和是跨年累计产量，不是单年产量。增长因子[1.10,0.95,1.02]表示+10%、-5%、+2%的逐年模拟变化，累计增长不是三项百分数相加。

In [58]:
table = np.array([[10, 20], [12, 22], [14, 24]])
print('各品种按年累计：\n', np.cumsum(table, axis=0))
print('不指定axis会展平：', np.cumsum(table))
factors = np.array([1.10, 0.95, 1.02])
print('累计增长百分数：', (np.cumprod(factors) - 1) * 100)

各品种按年累计：
 [[10 20]
 [22 42]
 [36 66]]
不指定axis会展平： [ 10  30  42  64  78 102]
累计增长百分数： [10.    4.5   6.59]


### 补充：整数溢出与精度选择（拓展）

累计积的整数运算可能按固定宽度回绕且不报错；int64不是无限大。下面显式指定int8只为演示；dtype=object交给Python整数可以保留任意大整数，但速度和存储方式不同。金额统计也需先明确精度和单位。

In [59]:
small = np.array([100, 2, 2], dtype=np.int8)
print('指定int8累计积（已溢出）：', np.cumprod(small, dtype=np.int8))
print('指定int64累计积：', np.cumprod(small, dtype=np.int64))
big = np.array([10**10, 10**10], dtype=np.int64)
print('Python整数累计积：', np.cumprod(big, dtype=object))

指定int8累计积（已溢出）： [ 100  -56 -112]
指定int64累计积： [100 200 400]
Python整数累计积： [10000000000 100000000000000000000]


## 独立预测与课后解释

先遮住输出，任选两题提交预测→运行结果→差异解释；基础评价每题结果50%、解释50%，不改变课程正式考核权重。

1. 对`[3,1,2]`执行`np.sort(x)`和`x.sort()`，原数组和返回值分别是什么？
2. 形状(3,2)的年度×品种表，`sum(axis=0)`与`sum(axis=1)`各统计什么、结果是什么形状？
3. 对含0.5的数组使用`fmt='%d'`保存，为什么结果变了？它是NumPy 2.0不支持的操作吗？
4. 选做：同一随机种子更换Generator后，为什么不应直接对照教材旧随机数？含缺失值均值应附什么口径？

教师核对：结果见前面对应演示，解释需覆盖原地修改、被归约的轴、格式截断、随机算法或有效观测数量。课堂使用及学生掌握程度等教师反馈后另记。

官方资料：[随机Generator](https://numpy.org/doc/2.0/reference/random/new-or-different.html)、[load](https://numpy.org/doc/2.0/reference/generated/numpy.load.html)、[savetxt](https://numpy.org/doc/2.0/reference/generated/numpy.savetxt.html)、[sort](https://numpy.org/doc/2.0/reference/generated/numpy.sort.html)、[lexsort](https://numpy.org/doc/2.0/reference/generated/numpy.lexsort.html)、[unique](https://numpy.org/doc/2.0/reference/generated/numpy.unique.html)、[std](https://numpy.org/doc/2.0/reference/generated/numpy.std.html)、[cumprod](https://numpy.org/doc/2.0/reference/generated/numpy.cumprod.html)。